# When enterprise data disagrees — Fabric companion
Daniele Mario Areddu · Data Saturday Sofia 2026

A small, explainable reconciliation flow for **company entities**. Record and review-evidence provenance is documented in asset_sources.md. CSV/JSON files represent extracts; this notebook does not connect to live CRM, ERP or portal APIs. Run after the talk, not as a live stage demo.

**Setup:** attach a disposable/default Lakehouse. Upload the contents of `datasets` into `Files/reconciliation_demo/datasets`. This notebook writes only tables beginning with `recon_demo_`. It overwrites derived demo outputs on each run. Use an isolated demonstration Lakehouse.

The rules are checked by the local reference (`reference.py`) and its tests. This notebook has not been run in a Fabric workspace as part of this repository's validation. Validate the first run in your workspace before publishing results or screenshots from it.

The final group step collects nine records to the driver. It deliberately shows group conflict handling; replace it for production scale. Table prefixes illustrate medallion roles inside one Lakehouse, not security boundaries.

## Messina context
The original project is an operational Python/FastAPI, PostgreSQL/PostGIS and React/TypeScript system. The Fabric customer fixture below is a separate teaching example. The named external Messina feeds are a reference architecture, not an asserted original integration inventory.

`messina_public_sample.csv` contains genuine municipal open-data addresses. `messina_example_records.csv` contains street-link records and states. These files are not inputs to the customer flow below. See `asset_sources.md` for attribution and original source links. The Microsoft slide screenshots show documentation interfaces, not a run of this notebook.


## Follow the slide records
C101/E501/P901 are Alfa; C102/E502 are the two Aurora companies; C103/E503 are the approved Delta pair; P902 stays held; C104 is Nova.
There are 26 possible cross-source pairs (4×3 + 4×2 + 3×2); the candidate rules keep seven. Candidate selection does not approve links. `expected/B001_results.json` and `expected/B002_results.json` hold the complete outputs of the local reference for comparison.
The Messina review CSV and the A–B–C group CSV are teaching views, not inputs to this customer flow.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T, Window
from delta.tables import DeltaTable
import csv, json, hashlib, itertools, re
from pathlib import Path
spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
BASE = Path('/lakehouse/default/Files/reconciliation_demo')
PREFIX = 'recon_demo_'
SCHEMA = ''  # default attached Lakehouse namespace; set e.g. 'dbo' if required
RAW_FIELDS = ['record_id','company_name','business_id','address','postcode','city','source_updated_at']
RAW_SCHEMA = T.StructType([T.StructField(c, T.StringType(), True) for c in RAW_FIELDS])
def table_name(role):
    name = PREFIX + role
    return SCHEMA + '.' + name if SCHEMA else name

def write_demo(df, role):
    df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(table_name(role))

## 1 · The same explicit rules as the local reference
This cell embeds the standalone reference implementation. No package installation is needed. The scalar fingerprint is used from Spark; the group function is used only for the tiny final consolidation. Name/address rules are deliberately limited to the limited ASCII forms. They are not a universal international address parser.

In [ ]:
"""Small, auditable reference implementation. Standard library only; not a scale design."""
import csv
import hashlib
import itertools
import json
import re
from pathlib import Path

RULE_VERSION = 'recon-demo-v2-en'
COMPARE_FIELDS = ('source_key', 'company_name', 'business_id', 'address', 'postcode', 'city')


def norm_name(value):
    return ' '.join(re.sub(r'[^a-z0-9 ]', '', (value or '').lower()).split())


def norm_address(value):
    value = re.sub(r'\bst\.(?=\s|,|$)', 'street', (value or '').lower().strip())
    return ' '.join(re.sub(r'[^a-z0-9 ]', '', value).split())


def usable_id(value):
    value = (value or '').strip().upper()
    return value if re.fullmatch(r'DEMO-[0-9]{3}', value) else ''


def normalise(record):
    r = dict(record)
    r['source_key'] = r['source'] + ':' + r['record_id']
    r.update(name_norm=norm_name(r['company_name']), address_norm=norm_address(r['address']),
             postcode_norm=(r.get('postcode') or '').strip().upper(),
             usable_id=usable_id(r.get('business_id')))
    return r


def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def name_similarity(a, b):
    return 1 - edit_distance(a, b) / max(len(a), len(b)) if a and b else 0.0


def candidate(a, b):
    if a['source'] == b['source']:
        return False
    exact = bool(a['usable_id'] and a['usable_id'] == b['usable_id'])
    block = bool(a['postcode_norm'] and a['postcode_norm'] == b['postcode_norm']
                 and a['name_norm'] and b['name_norm'] and a['name_norm'][:3] == b['name_norm'][:3])
    return exact or block


def evidence_fingerprint(a, b):
    items = sorted((a, b), key=lambda r: r['source_key'])
    payload = {'rule_version': RULE_VERSION, 'records': [
        {key: (r.get(key) or '') for key in COMPARE_FIELDS} for r in items]}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


def score_pair(a, b):
    a, b = sorted((a, b), key=lambda r: r['source_key'])
    sim = name_similarity(a['name_norm'], b['name_norm'])
    address = bool(a['address_norm'] and a['address_norm'] == b['address_norm'])
    postcode = bool(a['postcode_norm'] and a['postcode_norm'] == b['postcode_norm'])
    exact = bool(a['usable_id'] and a['usable_id'] == b['usable_id'])
    conflict = bool(a['usable_id'] and b['usable_id'] and a['usable_id'] != b['usable_id'])
    score = 60 * sim + 25 * address + 15 * postcode
    route = 'REJECT' if conflict else 'AUTO' if exact else 'REVIEW' if score >= 70 else 'KEEP_SEPARATE'
    return {'pair_key': a['source_key'] + '|' + b['source_key'], 'a_key': a['source_key'],
            'b_key': b['source_key'], 'name_similarity': sim, 'same_address': address,
            'same_postcode': postcode, 'exact_id': exact, 'id_conflict': conflict,
            'score': round(score, 6), 'initial_route': route, 'route': route,
            'reason': 'ID_CONFLICT' if conflict else 'EXACT_DEMO_ID' if exact else 'REVIEW_SCORE' if route == 'REVIEW' else 'WEAK_EVIDENCE',
            'evidence_fingerprint': evidence_fingerprint(a, b), 'rule_version': RULE_VERSION}


def apply_reviews(pairs, reviews):
    by_key = {r['pair_key']: r for r in reviews}
    out = []
    for p in pairs:
        p = dict(p)
        review = by_key.get(p['pair_key'])
        if review and p['initial_route'] == 'REVIEW':
            valid = (review['rule_version'] == RULE_VERSION and
                     review['evidence_fingerprint'] == p['evidence_fingerprint'] and
                     bool(review.get('reviewer') and review.get('evidence_reference')))
            if not valid:
                p['reason'] = 'STALE_OR_INCOMPLETE_REVIEW'
            elif review['decision'] == 'APPROVE':
                p.update(route='APPROVED', reason='MANUAL_EVIDENCE', evidence_reference=review['evidence_reference'])
            elif review['decision'] == 'REJECT':
                p.update(route='REJECT', reason='MANUAL_REJECTION')
        out.append(p)
    return out


def consolidate(records, pairs):
    """Driver grouping for nine rows only. Check whole groups before each union."""
    records = {r['source_key']: r for r in records}
    parent = {k: k for k in records}
    def root(k):
        while parent[k] != k:
            parent[k] = parent[parent[k]]
            k = parent[k]
        return k
    def members(k):
        return {x for x in records if root(x) == root(k)}
    denied = {frozenset((p['a_key'], p['b_key'])) for p in pairs if p['route'] == 'REJECT'}
    pairs = [dict(p) for p in pairs]
    for p in sorted(pairs, key=lambda p: (p['route'] != 'AUTO', p['pair_key'])):
        if p['route'] not in ('AUTO', 'APPROVED'):
            continue
        left, right = members(p['a_key']), members(p['b_key'])
        union = left | right
        ids = {records[k]['usable_id'] for k in union if records[k]['usable_id']}
        blocked = len(ids) > 1 or any(d <= union for d in denied)
        if blocked:
            p.update(route='GROUP_CONFLICT', reason='GROUP_CONTRADICTION')
        else:
            parent[root(p['b_key'])] = root(p['a_key'])
    held = set()
    for p in pairs:
        if p['route'] not in ('REVIEW', 'GROUP_CONFLICT'):
            continue
        left, right = members(p['a_key']), members(p['b_key'])
        if left == right:
            continue
        # Preserve a supported existing group; hold the new unresolved singleton.
        if len(left) > 1 and len(right) == 1:
            held |= right
        elif len(right) > 1 and len(left) == 1:
            held |= left
        else:
            held |= left | right
    groups = {}
    for key in records:
        if key not in held:
            groups.setdefault(root(key), []).append(records[key])
    gold, links = [], []
    for rows in groups.values():
        rows.sort(key=lambda r: r['source_key'])
        ids = {r['usable_id'] for r in rows if r['usable_id']}
        identity_anchor = 'business:' + next(iter(ids)) if ids else 'source:' + rows[0]['source_key']
        customer_id = 'CUS-' + hashlib.sha256(identity_anchor.encode()).hexdigest()[:12]
        def choose(field, priority):
            ordered = sorted(rows, key=lambda r: (priority.index(r['source']), r['source_key']))
            return next((r for r in ordered if r.get(field)), ordered[0])
        name = choose('company_name', ['CRM', 'ERP', 'PORTAL'])
        address = choose('address', ['ERP', 'CRM', 'PORTAL'])
        gold.append({'customer_id': customer_id, 'business_id': next(iter(ids)) if ids else '',
                     'display_name': name['company_name'], 'billing_address': address['address'],
                     'name_source_key': name['source_key'], 'billing_address_source_key': address['source_key'],
                     'rule_version': RULE_VERSION})
        links.extend({'source_key': r['source_key'], 'customer_id': customer_id,
                      'source_updated_at': r['source_updated_at'], 'batch_id': r['batch_id']} for r in rows)
    return {'pairs': pairs, 'gold_customers': sorted(gold, key=lambda r: r['customer_id']),
            'gold_source_links': sorted(links, key=lambda r: r['source_key']), 'held_source_keys': sorted(held)}


def evaluate(records, reviews=()):
    records = [normalise(r) for r in records]
    pairs = [score_pair(a, b) for a, b in itertools.combinations(records, 2) if candidate(a, b)]
    result = consolidate(records, apply_reviews(pairs, reviews))
    result['silver_records'] = records
    result['metrics'] = {'input_records': len(records), 'candidate_pairs': len(pairs),
                         'automatic_pairs': sum(p['initial_route'] == 'AUTO' for p in pairs),
                         'initial_review_pairs': sum(p['initial_route'] == 'REVIEW' for p in pairs),
                         'conflicting_pairs': sum(p['id_conflict'] for p in pairs),
                         'approved_review_pairs': sum(p['route'] == 'APPROVED' for p in result['pairs']),
                         'published_entities': len(result['gold_customers']),
                         'published_source_links': len(result['gold_source_links']),
                         'held_records': len(result['held_source_keys'])}
    return result


def load_inputs(folder):
    folder = Path(folder)
    rows = []
    for source, name in [('CRM', 'crm.csv'), ('ERP', 'erp.csv')]:
        with (folder / name).open(newline='') as f:
            rows.extend(dict(r, source=source, batch_id='B001', ingested_at='2026-10-08T09:00:00Z') for r in csv.DictReader(f))
    rows.extend(dict(r, source='PORTAL', batch_id='B001', ingested_at='2026-10-08T09:00:00Z') for r in json.loads((folder / 'portal.json').read_text()))
    return rows


def upsert_snapshot(history, incoming):
    snapshots = {(r['source'], r['record_id'], r['batch_id']): dict(r) for r in history}
    for r in incoming:
        key = (r['source'], r['record_id'], r['batch_id'])
        if key in snapshots and snapshots[key] != r:
            raise ValueError('A batch/source key cannot be reused for different evidence')
        snapshots[key] = dict(r)
    return list(snapshots.values())


def latest_snapshot(history):
    out = {}
    for r in sorted(history, key=lambda r: (r['source_updated_at'], r['batch_id'])):
        out[(r['source'], r['record_id'])] = r
    return list(out.values())


def checks(rows, reviews):
    result = evaluate(rows, reviews)
    assert result['metrics'] == dict(input_records=9, candidate_pairs=7, automatic_pairs=3,
        initial_review_pairs=3, conflicting_pairs=1, approved_review_pairs=1,
        published_entities=5, published_source_links=8, held_records=1)
    assert result['held_source_keys'] == ['PORTAL:P902']
    aurora = next(p for p in result['pairs'] if p['pair_key'] == 'CRM:C102|ERP:E502')
    assert aurora['score'] == 100 and aurora['route'] == 'REJECT'
    a = normalise(dict(rows[0], business_id=''))
    b = normalise(dict(rows[0], source='ERP', record_id='TEST', business_id=''))
    assert not score_pair(a, b)['exact_id']  # two missing identifiers are not evidence
    assert usable_id('BAD-ID') == '' and usable_id('DEMO-001') == 'DEMO-001'
    stale = [dict(r, evidence_fingerprint='stale') for r in reviews]
    assert evaluate(rows, stale)['metrics']['approved_review_pairs'] == 0
    # A-B and B-C cannot bypass an A-C identifier contradiction.
    bridge = [normalise(dict(rows[0], source='CRM', record_id='A', business_id='DEMO-001')),
              normalise(dict(rows[0], source='ERP', record_id='B', business_id='')),
              normalise(dict(rows[0], source='PORTAL', record_id='C', business_id='DEMO-009'))]
    edges = [dict(score_pair(bridge[0], bridge[1]), route='APPROVED'),
             dict(score_pair(bridge[1], bridge[2]), route='APPROVED')]
    guarded = consolidate(bridge, edges)
    assert any(p['route'] == 'GROUP_CONFLICT' for p in guarded['pairs'])
    history = upsert_snapshot([], rows)
    assert len(upsert_snapshot(history, rows)) == 9
    changed = dict(next(r for r in rows if r['source'] == 'ERP' and r['record_id'] == 'E501'),
                   address='Roma Street 14', source_updated_at='2026-10-09T08:00:00Z',
                   batch_id='B002', ingested_at='2026-10-09T09:00:00Z')
    history = upsert_snapshot(history, [changed])
    after = evaluate(latest_snapshot(history), reviews)
    before_id = next(r['customer_id'] for r in result['gold_customers'] if r['business_id'] == 'DEMO-001')
    after_alfa = next(r for r in after['gold_customers'] if r['business_id'] == 'DEMO-001')
    assert after_alfa['customer_id'] == before_id and after_alfa['billing_address'] == 'Roma Street 14'
    assert len(upsert_snapshot(history, [changed])) == 10
    return result, after

## 2 · Load extracts and preserve Bronze snapshots
The source key includes its namespace. Batch + source key identifies a snapshot. A rerun does not duplicate a snapshot. Reusing a batch/key for different source evidence raises an error rather than silently replacing history.

The original files remain in `Files`; Bronze retains original fields and acquisition metadata. Delta MERGE here is insert-only.

In [ ]:
def read_extract(name, source, batch):
    location = str(BASE / 'datasets' / name)
    if name.endswith('.json'):
        df = spark.read.schema(RAW_SCHEMA).option('multiLine', True).json(location)
    else:
        df = spark.read.schema(RAW_SCHEMA).option('header', True).csv(location)
    return (df.withColumn('source', F.lit(source))
            .withColumn('source_key', F.concat_ws(':', F.lit(source), F.col('record_id')))
            .withColumn('batch_id', F.lit(batch))
            .withColumn('ingested_at', F.current_timestamp())
            .withColumn('payload_hash', F.sha2(F.to_json(F.struct(*[F.col(c) for c in RAW_FIELDS])), 256)))

def ingest(df):
    if df.filter(F.col('record_id').isNull() | (F.trim('record_id') == '')).limit(1).count():
        raise ValueError('A source record key is required')
    if df.groupBy('source_key','batch_id').count().filter('count > 1').limit(1).count():
        raise ValueError('Duplicate source key in a batch')
    target = table_name('bronze_snapshots')
    if not spark.catalog.tableExists(target):
        df.write.format('delta').saveAsTable(target)
    else:
        collisions = (df.alias('new').join(spark.table(target).alias('old'), ['source_key','batch_id'])
                      .filter(F.col('new.payload_hash') != F.col('old.payload_hash')))
        if collisions.limit(1).count():
            raise ValueError('Existing batch/key has different source evidence; use a new batch')
        (DeltaTable.forName(spark, target).alias('old').merge(df.alias('new'),
            'old.source_key = new.source_key AND old.batch_id = new.batch_id')
            .whenNotMatchedInsertAll().execute())

initial = (read_extract('crm.csv','CRM','B001')
           .unionByName(read_extract('erp.csv','ERP','B001'))
           .unionByName(read_extract('portal.json','PORTAL','B001')))
ingest(initial)
ingest(initial)  # same batch/key: no extra snapshots
assert spark.table(table_name('bronze_snapshots')).filter("batch_id = 'B001'").count() == 9

## 3 · Silver makes fields comparable
Keep original fields. Validate the identifier format, not real-world ownership. Two missing IDs are not matching evidence. In this fixture, usable DEMO IDs are stipulated reliable company identifiers; a real implementation must verify source reliability.

Latest version is selected by source update time and batch. A production tie-break policy, deletes and late-arriving changes require explicit design.

In [ ]:
def build_silver():
    history = spark.table(table_name('bronze_snapshots'))
    ordered = Window.partitionBy('source_key').orderBy(
        F.to_timestamp('source_updated_at').desc_nulls_last(), F.col('batch_id').desc())
    latest = history.withColumn('_rank',F.row_number().over(ordered)).filter('_rank = 1').drop('_rank')
    def compact(column):
        return F.trim(F.regexp_replace(F.regexp_replace(column, r'[^a-z0-9 ]',''), r'\s+',' '))
    name = compact(F.lower(F.coalesce(F.col('company_name'),F.lit(''))))
    address = compact(F.regexp_replace(F.lower(F.trim(F.coalesce(F.col('address'),F.lit('')))),r'\bst\.(?=\s|,|$)','street'))
    raw_id = F.upper(F.trim(F.coalesce(F.col('business_id'),F.lit(''))))
    silver = (latest.withColumn('name_norm',name).withColumn('address_norm',address)
              .withColumn('postcode_norm',F.upper(F.trim(F.coalesce(F.col('postcode'),F.lit('')))))
              .withColumn('usable_id',F.when(raw_id.rlike(r'^DEMO-[0-9]{3}$'),raw_id).otherwise(F.lit(''))))
    write_demo(silver,'silver_records')
    exceptions = (silver.filter(F.col('usable_id') == '')
                  .select('source_key','business_id','batch_id',
                    F.when(raw_id == '',F.lit('MISSING_ID')).otherwise(F.lit('MALFORMED_ID')).alias('exception_type')))
    write_demo(exceptions,'exceptions')
    return silver

## 4 · Generate candidates before scoring
Cross-source pairs only, using either a usable exact company ID or postcode + three-letter normalised name prefix. A blocking rule can miss true matches; monitor its coverage independently.

Names use normalised Levenshtein similarity. Identifier conflicts override score. Exact reliable DEMO IDs can link automatically despite an old address. Scores of 70+ without exact identity evidence go to review; they do not auto-merge.

In [ ]:
fingerprint_udf = F.udf(lambda a,b: evidence_fingerprint(a.asDict(), b.asDict()), T.StringType())

def build_pairs(silver):
    a, b = silver.alias('a'), silver.alias('b')
    exact = (F.col('a.usable_id') != '') & (F.col('a.usable_id') == F.col('b.usable_id'))
    block = ((F.col('a.postcode_norm') != '') & (F.col('a.postcode_norm') == F.col('b.postcode_norm')) &
             (F.col('a.name_norm') != '') & (F.col('b.name_norm') != '') &
             (F.substring('a.name_norm',1,3) == F.substring('b.name_norm',1,3)))
    common = (F.col('a.source') != F.col('b.source')) & (F.col('a.source_key') < F.col('b.source_key'))
    # Separate keyed joins let Spark use the equality keys; avoid a single OR join.
    def project(joined):
        return joined.select(
        F.col('a.source_key').alias('a_key'), F.col('b.source_key').alias('b_key'),
        F.struct(*[F.col('a.'+c).alias(c) for c in COMPARE_FIELDS]).alias('a_record'),
        F.struct(*[F.col('b.'+c).alias(c) for c in COMPARE_FIELDS]).alias('b_record'),
        F.col('a.name_norm').alias('a_name'), F.col('b.name_norm').alias('b_name'),
        exact.alias('exact_id'),
        ((F.col('a.usable_id') != '') & (F.col('b.usable_id') != '') &
         (F.col('a.usable_id') != F.col('b.usable_id'))).alias('id_conflict'),
        ((F.col('a.address_norm') != '') & (F.col('a.address_norm') == F.col('b.address_norm'))).alias('same_address'),
        ((F.col('a.postcode_norm') != '') & (F.col('a.postcode_norm') == F.col('b.postcode_norm'))).alias('same_postcode'))
    pairs = project(a.join(b,common & exact)).unionByName(project(a.join(b,common & block))).dropDuplicates(['a_key','b_key'])
    max_len = F.greatest(F.length('a_name'),F.length('b_name'))
    pairs = (pairs.withColumn('pair_key',F.concat_ws('|','a_key','b_key'))
             .withColumn('name_similarity',F.when((F.col('a_name') != '') & (F.col('b_name') != ''),
                1.0 - F.levenshtein('a_name','b_name') / max_len).otherwise(F.lit(0.0)))
             .withColumn('score',F.round(60*F.col('name_similarity') + 25*F.col('same_address').cast('int') +
                                         15*F.col('same_postcode').cast('int'),6))
             .withColumn('initial_route',F.when(F.col('id_conflict'),'REJECT').when(F.col('exact_id'),'AUTO')
                         .when(F.col('score') >= 70,'REVIEW').otherwise('KEEP_SEPARATE'))
             .withColumn('reason',F.when(F.col('id_conflict'),'ID_CONFLICT').when(F.col('exact_id'),'EXACT_DEMO_ID')
                         .when(F.col('score') >= 70,'REVIEW_SCORE').otherwise('WEAK_EVIDENCE'))
             .withColumn('evidence_fingerprint',fingerprint_udf('a_record','b_record'))
             .withColumn('rule_version',F.lit(RULE_VERSION)))
    return pairs

## 5 · A review decision adds evidence
The reviewer checks `ONB-DELTA-42`. Approval applies only to CRM C103 + ERP E503. It does not approve the Delta portal records. Reuse a decision only when the evidence fingerprint and rule version are unchanged. Store the review record separately from derived match output.

In [ ]:
REVIEW_FIELDS = ['pair_key','decision','reviewer','reviewed_at','evidence_reference','rule_version','evidence_fingerprint']
review_schema = T.StructType([T.StructField(c,T.StringType(),True) for c in REVIEW_FIELDS])
reviews = spark.read.schema(review_schema).option('header',True).csv(str(BASE/'datasets/review_decisions.csv'))
assert reviews.count() == reviews.dropDuplicates(['pair_key']).count(), 'One current decision per pair is required'
write_demo(reviews,'review_decisions')

def route_with_reviews(pairs):
    decision = (reviews.withColumnRenamed('rule_version','review_rule_version')
                .withColumnRenamed('evidence_fingerprint','review_fingerprint'))
    p = pairs.join(decision,'pair_key','left')
    valid = F.coalesce((F.col('review_fingerprint') == F.col('evidence_fingerprint')) &
                       (F.col('review_rule_version') == RULE_VERSION) &
                       (F.length(F.trim(F.coalesce(F.col('reviewer'),F.lit('')))) > 0) &
                       (F.length(F.trim(F.coalesce(F.col('evidence_reference'),F.lit('')))) > 0),F.lit(False))
    pending = F.col('initial_route') == 'REVIEW'
    return (p.withColumn('route',F.when(pending & valid & (F.col('decision') == 'APPROVE'),'APPROVED')
              .when(pending & valid & (F.col('decision') == 'REJECT'),'REJECT').otherwise(F.col('initial_route')))
            .withColumn('reason',F.when(pending & valid & (F.col('decision') == 'APPROVE'),'MANUAL_EVIDENCE')
              .when(pending & valid & (F.col('decision') == 'REJECT'),'MANUAL_REJECTION')
              .when(pending & F.col('decision').isNotNull() & ~valid,'STALE_OR_INCOMPLETE_REVIEW').otherwise(F.col('reason'))))

## 6 · Check whole groups, then choose values by field
An A–B–C path is not sufficient if the combined group contains contradictory usable company IDs or an explicit rejection. The driver grouping below checks the union before each accepted edge.

An unresolved singleton near an already supported group is held; the supported group can publish. Other unresolved groups are held. This is an explicit demo publication policy, not a universal rule.

Display name prefers CRM; billing address prefers ERP. Each chosen value retains a source key. Deterministic hash IDs stay stable for this fixture's address update. A persistent identity registry is required for general identifier corrections, introductions, merges and splits.

In [ ]:
PAIR_FIELDS = ['pair_key','a_key','b_key','name_similarity','same_address','same_postcode','exact_id','id_conflict',
               'score','initial_route','route','reason','evidence_fingerprint','rule_version','evidence_reference']
boolean_fields = {'same_address','same_postcode','exact_id','id_conflict'}
pair_schema = T.StructType([T.StructField(c,T.BooleanType() if c in boolean_fields else
    T.DoubleType() if c in {'score','name_similarity'} else T.StringType(),True) for c in PAIR_FIELDS])
GOLD_FIELDS = ['customer_id','business_id','display_name','billing_address','name_source_key','billing_address_source_key','rule_version']
LINK_FIELDS = ['source_key','customer_id','source_updated_at','batch_id']
def strings_schema(columns): return T.StructType([T.StructField(c,T.StringType(),True) for c in columns])

def run_reconciliation(run_id):
    silver = build_silver()
    routed = route_with_reviews(build_pairs(silver))
    tiny_records = [r.asDict(recursive=True) for r in silver.collect()]
    tiny_pairs = [r.asDict(recursive=True) for r in routed.collect()]
    result = consolidate(tiny_records,tiny_pairs)  # teaching shortcut, not a production-scale operation
    pair_output = spark.createDataFrame(result['pairs'],pair_schema)
    write_demo(pair_output,'match_pairs')
    write_demo(pair_output.filter(F.col('route').isin('REVIEW','GROUP_CONFLICT')),'review_queue')
    write_demo(spark.createDataFrame(result['gold_customers'],strings_schema(GOLD_FIELDS)),'gold_customers')
    write_demo(spark.createDataFrame(result['gold_source_links'],strings_schema(LINK_FIELDS)),'gold_source_links')
    write_demo(spark.createDataFrame([{'source_key':k,'status':'PENDING_REVIEW'} for k in result['held_source_keys']],
                                    strings_schema(['source_key','status'])),'held_records')
    metrics = {'run_id':run_id,'input_records':len(tiny_records),'candidate_pairs':len(tiny_pairs),
       'automatic_pairs':sum(p['initial_route']=='AUTO' for p in tiny_pairs),
       'initial_review_pairs':sum(p['initial_route']=='REVIEW' for p in tiny_pairs),
       'conflicting_pairs':sum(p['id_conflict'] for p in tiny_pairs),
       'approved_review_pairs':sum(p['route']=='APPROVED' for p in result['pairs']),
       'published_entities':len(result['gold_customers']),'published_source_links':len(result['gold_source_links']),
       'held_records':len(result['held_source_keys'])}
    metric_schema = T.StructType([T.StructField(c,T.StringType() if c=='run_id' else T.LongType(),False) for c in metrics])
    metric_df = spark.createDataFrame([metrics],metric_schema)
    # Preserve a compact run history. Derived tables above represent the current publication.
    target=table_name('run_metrics')
    if spark.catalog.tableExists(target):
        (DeltaTable.forName(spark,target).alias('old').merge(metric_df.alias('new'),'old.run_id = new.run_id')
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    else:
        metric_df.write.format('delta').saveAsTable(target)
    return result,metrics

baseline, metrics_B001 = run_reconciliation('B001')
assert {k:v for k,v in metrics_B001.items() if k!='run_id'} == {
 'input_records':9,'candidate_pairs':7,'automatic_pairs':3,'initial_review_pairs':3,'conflicting_pairs':1,
 'approved_review_pairs':1,'published_entities':5,'published_source_links':8,'held_records':1}
assert baseline['held_source_keys'] == ['PORTAL:P902']
print(metrics_B001)
spark.table(table_name('gold_customers')).show(truncate=False)
spark.table(table_name('review_queue')).select('pair_key','score','reason').show(truncate=False)

## 7 · Count outcomes and evaluate labelled pairs separately
Counts describe the run; they do not prove accuracy. This tiny labelled fixture has five known pairs and two UNKNOWN portal pairs. Exclude UNKNOWN from accuracy calculations rather than pretending the missing review has an answer. Its perfect labelled metrics do not estimate production accuracy.

In [ ]:
labels = (spark.read.option('header',True).csv(str(BASE/'datasets/labelled_pairs.csv'))
          .filter(F.col('label').isin('MATCH','DIFFERENT')))
predictions = spark.table(table_name('match_pairs')).select('pair_key','route')
check = labels.join(predictions,'pair_key','left')
check = (check.withColumn('accepted',F.coalesce(F.col('route').isin('AUTO','APPROVED'),F.lit(False)))
         .withColumn('positive',F.col('label')=='MATCH'))
true_positive = check.filter(F.col('accepted') & F.col('positive')).count()
false_positive = check.filter(F.col('accepted') & ~F.col('positive')).count()
false_negative = check.filter(~F.col('accepted') & F.col('positive')).count()
true_pairs = check.filter('positive').count()
covered_true = check.filter(F.col('positive') & F.col('route').isNotNull()).count()
print({'labelled_pairs':check.count(),'true_positive':true_positive,'false_positive':false_positive,
       'false_negative':false_negative,'blocking_recall':covered_true/true_pairs if true_pairs else None})
assert true_positive == 4 and false_positive == 0 and false_negative == 0

## 8 · Tomorrow: ingest one changed source record
B002 updates ERP E501's billing address. Bronze keeps both versions. The latest company snapshot still has nine records. Recompute the tiny full candidate set and publication; this is **incremental ingestion, not a complete incremental matching engine**.

The same trusted DEMO-001 anchor keeps Alfa's customer ID stable. The unchanged Delta review remains usable. A relevant Delta field change would make its fingerprint stale and return that pair to review.

For a clean B001 demonstration, use a fresh demo Lakehouse: after B002 has been ingested, the latest snapshot correctly remains B002 on reruns.

In [ ]:
alfa_before = next(r['customer_id'] for r in baseline['gold_customers'] if r['business_id']=='DEMO-001')
changed = read_extract('erp_B002.csv','ERP','B002')
ingest(changed)
ingest(changed)
after, metrics_B002 = run_reconciliation('B002')
alfa_after = next(r for r in after['gold_customers'] if r['business_id']=='DEMO-001')
assert alfa_after['customer_id'] == alfa_before
assert alfa_after['billing_address'] == 'Roma Street 14'
assert spark.table(table_name('bronze_snapshots')).count() == 10
assert metrics_B002['input_records'] == 9 and metrics_B002['published_entities'] == 5
print(metrics_B002)
spark.table(table_name('run_metrics')).orderBy('run_id').show(truncate=False)

## Publication and production extensions
Create a semantic model from the intended Gold tables explicitly; do not assume a default model is created automatically. Apply access controls and appropriate relationships. Application/API consumption requires a separately designed backend integration.

Before production: use a persistent entity registry; support correction and split workflows; design distributed grouping and affected-group reprocessing; monitor skew and candidate counts; version review decisions; define deletes, late arrivals and tie-breaks; calibrate evidence with independent labelled data; identify review owners and service targets.

Official references:
- [Medallion architecture](https://learn.microsoft.com/en-us/fabric/onelake/onelake-medallion-lakehouse-architecture)
- [Notebook activity in a pipeline](https://learn.microsoft.com/en-us/fabric/data-factory/notebook-activity)
- [Load Lakehouse data with notebooks](https://learn.microsoft.com/en-us/fabric/data-engineering/lakehouse-notebook-load-data)
- [Create a semantic model](https://learn.microsoft.com/en-us/fabric/data-warehouse/create-semantic-model)
- [Item lineage](https://learn.microsoft.com/en-us/fabric/governance/lineage)
- [PySpark Levenshtein](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.levenshtein.html)

## Slides and evidence files
The deck follows the same record keys. Slide 23 shows Nova in an excluded pair, slide 31 rechecks a saved decision before applying it, slide 34 separates source mappings from field provenance, and slide 36 counts the five published companies before discussing accuracy.

`datasets/teaching_evidence.json` describes what ONB-DELTA-42 and REF-CELI-01 assert, and what they do not. The notebook does not read it, and it adds no customer rows. The review cell reads `datasets/review_decisions.csv`, whose fingerprint matches rule version `recon-demo-v2-en` and the current English records.